In [1]:
%pwd

'/mnt/1A3499713499511D/OLD_data30-1-26/MLE2025/Faceit'

In [2]:
import polars as pl
import pymongoarrow.api as pmi
from pipeline.orch import getdata
import pyarrow.parquet as pq
import polars as pl

ratingpath =  "data/raw/ratings.parquet"
elopath =  "data/raw/matches_elo.parquet"
lifetimepath =  "data/raw/lifetime.parquet"
lfscorespath =  "data/raw/lfscores.parquet"

rating_df = pl.scan_parquet(ratingpath)
elo_df = pl.scan_parquet(elopath)
#lifetime_df = pl.scan_parquet(lifetimepath)
lfscores_df = pl.scan_parquet(lfscorespath)

In [ ]:
references = {
    'ratings': [rating_df, ratingpath],
    'elo': [elo_df, elopath],
    'lfscores': [lfscores_df, lfscorespath],
    #'lifetime': [lifetime_df, lifetimepath]
}


## ELO PATH

In [4]:
df = pl.scan_parquet(elopath)

In [5]:
struct_removal = [col for col, dtype in zip(df.columns, df.dtypes) if dtype == pl.Struct and not col == 'results']
elo_dropped = df.drop(struct_removal)

faction1_col = df.select('teams').unnest('teams').select('faction1').unnest('faction1')
faction1_renamed = faction1_col.rename({
    col: f"{col}_1" for col in faction1_col.columns
}).unnest('stats_1').rename({'rating': 'rating_1'})

faction2_col = df.select('teams').unnest('teams').select('faction2').unnest('faction2')
faction2_renamed = faction2_col.rename({
    col: f"{col}_2" for col in faction2_col.columns
}).unnest('stats_2').rename({'rating': 'rating_2'})

voting_col = df.select('voting').unnest('voting').unnest('map').explode('pick')

final_elo = pl.concat([elo_dropped, faction1_renamed, faction2_renamed, voting_col], how = 'horizontal').drop('detailed_results')
final_elo.sink_parquet('data/interim/silver_matches_elo.parquet')


/tmp/ipykernel_13595/3874875722.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  struct_removal = [col for col, dtype in zip(df.columns, df.dtypes) if dtype == pl.Struct and not col == 'results']
/tmp/ipykernel_13595/3874875722.py:1: PerformanceWarning: Determining the data types of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().dtypes()` to get the data types without this warning.
  struct_removal = [col for col, dtype in zip(df.columns, df.dtypes) if dtype == pl.Struct and not col == 'results']
/tmp/ipykernel_13595/3874875722.py:6: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names

## RATING PATH

In [6]:
# Keep _id alongside team_agg before selecting
temp_rating = rating_df.drop(['stageId', 'players'])  # _id still here

# Unnest team_agg separately
team_agg_unnested = temp_rating.select('team_agg').unnest('team_agg')

faction1_col = team_agg_unnested.select('faction1').unnest('faction1')
faction1_renamed = faction1_col.rename({col: f"{col}_1" for col in faction1_col.columns})

faction2_col = team_agg_unnested.select('faction2').unnest('faction2')
faction2_renamed = faction2_col.rename({col: f"{col}_2" for col in faction2_col.columns})

# Include the base df (which has _id), drop the struct columns after
base = temp_rating.drop(['team_agg'])  # just _id (and anything else you kept)

final_rating = pl.concat(
    [base, faction1_renamed, faction2_renamed],
    how='horizontal'
)

final_rating.sink_parquet('data/interim/silver_ratings.parquet')

/tmp/ipykernel_13595/4115110036.py:8: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  faction1_renamed = faction1_col.rename({col: f"{col}_1" for col in faction1_col.columns})
/tmp/ipykernel_13595/4115110036.py:11: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  faction2_renamed = faction2_col.rename({col: f"{col}_2" for col in faction2_col.columns})


## LFSCORES PATH

In [ ]:
import polars as pl

CS2_MAP_POOL = [
    'Mirage', 'Inferno', 'Nuke', 'Dust2',
    'Ancient', 'Anubis', 'Overpass',
    'Vertigo', 'Train', 'Cache'
]

df = pl.scan_parquet("data/raw/lfscores.parquet")
base = df.select('team_lifetime').unnest('team_lifetime')
root_id = df.select('_id')

frames = []

for faction_idx, faction_col in enumerate(['faction1', 'faction2'], start=1):
    for player_idx in range(5):
        player_num = player_idx + 1

        player = (
            base
            .select(pl.col(faction_col).list.get(player_idx))
            .unnest(faction_col)
        )

        # Drop Recent Results — it's List(String), can't flatten with scalars
        lifetime = (
            player
            .select('lifetime')
            .unnest('lifetime')
            .drop(['Recent Results'])
        )
        lifetime = lifetime.rename({
            col: f"lifetime_{col}"
            for col in lifetime.collect_schema().names()
        })

        # Filter to valid 5v5 map pool segments
        # Normalize field order explicitly so faction1 and faction2 structs match
        filtered_segments = player.select(
            pl.col('segments')
            .list.eval(
                pl.element().filter(
                    (pl.element().struct.field('mode') == '5v5') &
                    (pl.element().struct.field('label').is_in(CS2_MAP_POOL))
                )
            )
            .list.eval(
                pl.struct(
                    pl.element().struct.field('label').alias('label'),
                    pl.element().struct.field('mode').alias('mode'),
                    pl.element().struct.field('stats').alias('stats'),
                )
            )
            .alias('segments_filtered')
        )

        player_row = pl.concat([
            root_id.rename({'_id': 'match_id'}),
            player.select(['_id', 'game_id']).rename({'_id': 'player_id'}),
            lifetime,
            filtered_segments,
        ], how='horizontal').with_columns([
            pl.lit(faction_idx).alias('faction'),
            pl.lit(player_num).alias('player_num')
        ])

        frames.append(player_row)

# Collect — diagonal_relaxed handles the lifetime field differences between factions
combined = pl.concat(frames, how='diagonal_relaxed').collect()

# Drop players with no valid map segments
combined = combined.filter(pl.col('segments_filtered').list.len() > 0)

# Explode one row per player per map, then flatten
final = (
    combined
    .explode('segments_filtered')
    .unnest('segments_filtered')    # → label, mode, stats
    .rename({'label': 'map'})
    .drop('mode')                   # always '5v5', redundant now
    .unnest('stats')
)

# Prefix all stat columns
skip = {'match_id', 'player_id', 'game_id', 'faction', 'player_num', 'map', 'stageId'}
stat_cols = [c for c in final.columns if c not in skip and not c.startswith('lifetime_')]
final = final.rename({col: f"seg_{col}" for col in stat_cols})

final.sink_parquet('data/interim/silver_lfscores.parquet')

In [ ]:
import polars as pl
import os
import shutil

CS2_MAP_POOL = [
    'Mirage', 'Inferno', 'Nuke', 'Dust2',
    'Ancient', 'Anubis', 'Overpass',
    'Vertigo', 'Train', 'Cache'
]

BATCH_SIZE = 10_000  # ~15K docs × 10 players = 150K rows per batch, safe on 16GB
PARQUET_IN  = "data/raw/lfscores.parquet"
PARQUET_OUT = "data/interim/silver_lfscores.parquet"
BATCH_DIR   = "data/interim/_batches"

os.makedirs(BATCH_DIR, exist_ok=True)


def process_batch(df_lazy: pl.LazyFrame) -> pl.DataFrame:
    base     = df_lazy.select('team_lifetime').unnest('team_lifetime')
    root_id  = df_lazy.select('_id')
    frames   = []

    for faction_idx, faction_col in enumerate(['faction1', 'faction2'], start=1):
        for player_idx in range(5):
            player_num = player_idx + 1

            player = (
                base
                .select(pl.col(faction_col).list.get(player_idx, null_on_oob= True))
                .unnest(faction_col)
            )

            lifetime_raw  = (
                player
                .select('lifetime')
                .unnest('lifetime')
                .drop(['Recent Results'])
            )
            lifetime_cols = lifetime_raw.collect_schema().names()
            lifetime      = lifetime_raw.rename({c: f"lifetime_{c}" for c in lifetime_cols})

            filtered_segments = player.select(
                pl.col('segments')
                .list.eval(
                    pl.element().filter(
                        (pl.element().struct.field('mode') == '5v5') &
                        (pl.element().struct.field('label').is_in(CS2_MAP_POOL))
                    )
                )
                .list.eval(
                    pl.struct(
                        pl.element().struct.field('label').alias('label'),
                        pl.element().struct.field('mode').alias('mode'),
                        pl.element().struct.field('stats').alias('stats'),
                    )
                )
                .alias('segments_filtered')
            )

            player_row = (
                pl.concat([
                    root_id.rename({'_id': 'match_id'}),
                    player.select(['_id', 'game_id']).rename({'_id': 'player_id'}),
                    lifetime,
                    filtered_segments,
                ], how='horizontal')
                .with_columns([
                    pl.lit(faction_idx).alias('faction'),  # ✅ baked in before collect
                    pl.lit(player_num).alias('player_num'),
                ])
                .collect() # ✅ moved here
                                                
            )
            player_row = (
                pl.concat([
                    root_id.rename({'_id': 'match_id'}),
                    player.select(['_id', 'game_id']).rename({'_id': 'player_id'}),
                    lifetime,
                    filtered_segments,
                ], how='horizontal')
                .collect()
            )

            player_row = player_row.with_columns(
                pl.Series("faction", [faction_idx] * len(player_row), dtype=pl.Int8),
                pl.Series("player_num", [player_num] * len(player_row), dtype=pl.Int8),
            )

            frames.append(player_row)

    combined = (
        pl.concat(frames, how='diagonal_relaxed')
        .filter(pl.col('segments_filtered').list.len() > 0)
    )

    final = (
        combined
        .explode('segments_filtered')
        .unnest('segments_filtered')
        .rename({'label': 'map'})
        .drop('mode')
        .unnest('stats')
    )

    skip     = {'match_id', 'player_id', 'game_id', 'faction', 'player_num', 'map', 'stageId'}
    stat_cols = [c for c in final.columns if c not in skip and not c.startswith('lifetime_')]
    return final.rename({c: f"seg_{c}" for c in stat_cols})


# ── Main loop ────────────────────────────────────────────────────────────────

total = pl.scan_parquet(PARQUET_IN).select(pl.len()).collect().item()
print(f"Total documents: {total:,}  →  {(total // BATCH_SIZE) + 1} batches")

for batch_num, start in enumerate(range(0, total, BATCH_SIZE)):
    print(f"  Batch {batch_num:04d}  rows {start:,} – {min(start + BATCH_SIZE, total):,} ...", end=' ')

    batch_lazy   = pl.scan_parquet(PARQUET_IN).slice(start, BATCH_SIZE)
    batch_result = process_batch(batch_lazy)
    batch_result.write_parquet(f"{BATCH_DIR}/batch_{batch_num:04d}.parquet")

    del batch_result   # release memory immediately
    print("done")

# ── Merge all batches lazily — no RAM spike ──────────────────────────────────
print("Merging batches → final parquet ...")
pl.scan_parquet(f"{BATCH_DIR}/*.parquet").sink_parquet(PARQUET_OUT)

# ── Cleanup ──────────────────────────────────────────────────────────────────
shutil.rmtree(BATCH_DIR)
print(f"Done → {PARQUET_OUT}")

Total documents: 365,225  →  37 batches
  Batch 0000  rows 0 – 10,000 ... done
  Batch 0001  rows 10,000 – 20,000 ... done
  Batch 0002  rows 20,000 – 30,000 ... done
  Batch 0003  rows 30,000 – 40,000 ... done
  Batch 0004  rows 40,000 – 50,000 ... done
  Batch 0005  rows 50,000 – 60,000 ... done
  Batch 0006  rows 60,000 – 70,000 ... done
  Batch 0007  rows 70,000 – 80,000 ... done
  Batch 0008  rows 80,000 – 90,000 ... done
  Batch 0009  rows 90,000 – 100,000 ... done
  Batch 0010  rows 100,000 – 110,000 ... done
  Batch 0011  rows 110,000 – 120,000 ... done
  Batch 0012  rows 120,000 – 130,000 ... done
  Batch 0013  rows 130,000 – 140,000 ... done
  Batch 0014  rows 140,000 – 150,000 ... done
  Batch 0015  rows 150,000 – 160,000 ... done
  Batch 0016  rows 160,000 – 170,000 ... done
  Batch 0017  rows 170,000 – 180,000 ... done
  Batch 0018  rows 180,000 – 190,000 ... done
  Batch 0019  rows 190,000 – 200,000 ... done
  Batch 0020  rows 200,000 – 210,000 ... done
  Batch 0021  rows

In [1]:
silverlfscorespath = 'data/interim/silver_lfscores.parquet'
import polars as pl
df = pl.scan_parquet(silverlfscorespath)

In [7]:
import pandas as pd
import polars as pl

with pl.Config(tbl_rows=100, tbl_cols=200, tbl_width_chars=10000):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    display(pl.scan_parquet('data/interim/silver_lfscores.parquet').head(100).collect().to_pandas().sort_values(by = 'faction', ascending= False))

,match_id,player_id,game_id,lifetime_1v1 Win Rate,lifetime_1v2 Win Rate,lifetime_ADR,lifetime_Average Headshots %,lifetime_Average K/D Ratio,lifetime_Current Win Streak,lifetime_Enemies Flashed per Round,lifetime_Entry Rate,lifetime_Entry Success Rate,lifetime_Flash Success Rate,lifetime_Flashes per Round,lifetime_K/D Ratio,lifetime_Longest Win Streak,lifetime_Matches,lifetime_Sniper Kill Rate,lifetime_Sniper Kill Rate per Round,lifetime_Total 1v1 Count,lifetime_Total 1v1 Wins,lifetime_Total 1v2 Count,lifetime_Total 1v2 Wins,lifetime_Total Damage,lifetime_Total Enemies Flashed,lifetime_Total Entry Count,lifetime_Total Entry Wins,lifetime_Total Flash Count,lifetime_Total Flash Successes,lifetime_Total Headshots %,lifetime_Total Kills with extended stats,lifetime_Total Matches,lifetime_Total Rounds with extended stats,lifetime_Total Sniper Kills,lifetime_Total Utility Count,lifetime_Total Utility Damage,lifetime_Total Utility Successes,lifetime_Utility Damage Success Rate,lifetime_Utility Damage per Round,lifetime_Utility Success Rate,lifetime_Utility Usage per Round,lifetime_Win Rate %,lifetime_Wins,map,seg_1v1 Win Rate,seg_1v2 Win Rate,seg_ADR,seg_Assists,seg_Average Assists,seg_Average Deaths,seg_Average Headshots %,seg_Average K/D Ratio,seg_Average K/R Ratio,seg_Average Kills,seg_Average MVPs,seg_Average Penta Kills,seg_Average Quadro Kills,seg_Average Triple Kills,seg_Deaths,seg_Enemies Flashed per Round,seg_Entry Rate,seg_Entry Success Rate,seg_Flash Success Rate,seg_Flashes per Round,seg_Headshots,seg_Headshots per Match,seg_K/D Ratio,seg_K/R Ratio,seg_Kills,seg_MVPs,seg_Matches,seg_Penta Kills,seg_Quadro Kills,seg_Rounds,seg_Sniper Kill Rate,seg_Sniper Kill Rate per Round,seg_Total 1v1 Count,seg_Total 1v1 Wins,seg_Total 1v2 Count,seg_Total 1v2 Wins,seg_Total Damage,seg_Total Enemies Flashed,seg_Total Entry Count,seg_Total Entry Wins,seg_Total Flash Count,seg_Total Flash Successes,seg_Total Headshots %,seg_Total Kills with extended stats,seg_Total Matches,seg_Total Rounds with extended stats,seg_Total Sniper Kills,seg_Total Utility Count,seg_Total Utility Damage,seg_Total Utility Successes,seg_Triple Kills,seg_Utility Damage Success Rate,seg_Utility Damage per Round,seg_Utility Success Rate,seg_Utility Usage per Round,seg_Win Rate %,seg_Wins,faction,player_num
0,1-b3b89125-0263-417b-a51b-fe0b152cfcf9,663a71f4-97dc-4a40-a7ea-467c1e6cae05,cs2,0.3,0.18,74.43,42,1.61,0,0.56,0.15,0.53,0.52,0.73,899.71,24,559,0.03,0.04,33,10,39,7,105171,798,210,111,1026,529,23595,967,66,1413,43,423,7756,194,18.34,5.49,0.46,0.3,63,353,Inferno,0.5,0,74.8,28,4,13.29,55,1.17,0.71,14.29,1.86,0.14,0.14,0.14,93,0.29,0.16,0.57,0.35,0.6,57,8.14,8.18,4.95,100,13,7,1,1,142,0.03,0.04,4,2,1,0,10622,41,23,13,85,30,383,100,7,142,4,76,1458,42,1,19.18,10.27,0.55,0.54,57,4,1,1
1,1-b3b89125-0263-417b-a51b-fe0b152cfcf9,663a71f4-97dc-4a40-a7ea-467c1e6cae05,cs2,0.3,0.18,74.43,42,1.61,0,0.56,0.15,0.53,0.52,0.73,899.71,24,559,0.03,0.04,33,10,39,7,105171,798,210,111,1026,529,23595,967,66,1413,43,423,7756,194,18.34,5.49,0.46,0.3,63,353,Anubis,0,0,69.69,93,5.17,14.5,52,0.95,0.62,12.44,1.78,0,0.11,0.5,261,0.5,0.13,0.4,0.45,0.77,115,6.39,17.09,11.08,224,32,18,0,2,372,0.01,0.02,6,0,8,0,21743,155,40,16,240,108,945,186,15,312,3,82,1866,41,9,22.76,5.98,0.5,0.26,44,8,1,1
2,1-b3b89125-0263-417b-a51b-fe0b152cfcf9,663a71f4-97dc-4a40-a7ea-467c1e6cae05,cs2,0.3,0.18,74.43,42,1.61,0,0.56,0.15,0.53,0.52,0.73,899.71,24,559,0.03,0.04,33,10,39,7,105171,798,210,111,1026,529,23595,967,66,1413,43,423,7756,194,18.34,5.49,0.46,0.3,63,353,Vertigo,None,None,None,10,10,32,30,1.03,0.8,33,5,0,0,5,32,None,None,None,None,None,10,10,1.03,0.8,33,5,1,0,0,41,None,None,None,None,None,None,None,None,None,None,None,None,30,None,None,None,None,None,None,None,5,None,None,None,None,0,0,1,1
3,1-b3b89125-0263-417b-a51b-fe0b152cfcf9,663a71f4-97dc-4a40-a7ea-467c1e6cae05,cs2,0.3,0.18,74.43,42,1.61,0,0.56,0.15,0.53,0.52,0.73,899.71,24,559,0.03,0.04,33,10,39,7,105171,798,210,111,1026,529,23595,967,66,1413,43,423,7756

In [2]:
import pandas as pd

with pl.Config(tbl_rows=100, tbl_cols=200, tbl_width_chars=10000):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    display(pl.scan_parquet('data/interim/silver_matches_elo.parquet').head(100).collect().to_pandas())

,_id,match_id,started_at,finished_at,results,stageId,batchId,roster_1,rating_1,roster_2,rating_2,pick
0,1-4cb251f5-febf-41b1-808b-c94ae3882c74,1-4cb251f5-febf-41b1-808b-c94ae3882c74,1776027544,1776030425,"{'winner': 'faction1', 'score': {'faction1': 1...",stage_1_20260424_0830,NaN,[{'player_id': '663a71f4-97dc-4a40-a7ea-467c1e...,2336.0,[{'player_id': '7fe36329-672b-469a-b6cd-f82032...,2385.0,de_mirage
1,1-d8ccf3d9-326f-4656-853b-37fbd2dce3d5,1-d8ccf3d9-326f-4656-853b-37fbd2dce3d5,1772741372,1772743731,"{'winner': 'faction2', 'score': {'faction1': 8...",stage_1_20260424_0830,NaN,[{'player_id': 'ac15b261-1e4c-435c-9908-1130da...,2403.0,[{'player_id': '8f486899-633e-4766-ac37-beead8...,2409.0,de_nuke
2,1-3421873c-e712-4be3-bdc0-818a4f183bc7,1-3421873c-e712-4be3-bdc0-818a4f183bc7,1771875814,1771878564,"{'winner': 'faction2', 'score': {'faction1': 1...",stage_1_20260424_0830,NaN,[{'player_id': '9c0521d8-d323-41f2-91c3-3a79a3...,NaN,[{'player_id': '056232a7-6889-4a78-8ad8-c98f57...,NaN,de_inferno
3,1-1cb55b79-4ef3-4cfd-b89b-1cebeeab66ff,1-1cb55b79-4ef3-4cfd-b89b-1cebeeab66ff,1776976715,1776978321,"{'winner': 'faction2', 'score': {'faction1': 5...",stage_1_20260424_0830,NaN,[{'player_id': '7fa5f3c5-96fb-4e30-87bf-d84544...,2427.0,[{'player_id': '10da9dfa-5058-41f0-9aff-636f41...,2551.0,de_dust2
4,1-f53bea17-570a-4582-91aa-912e6e2352dc,1-f53bea17-570a-4582-91aa-912e6e2352dc,1772744130,1772747115,"{'winner': 'faction1', 'score': {'faction1': 1...",stage_1_20260424_0830,NaN,[{'player_id': '4d6ac2f7-de8f-4ee9-badc-0c1ce0...,2481.0,[{'player_id': 'ba205bd4-6671-45c4-bef6-f397e2...,2479.0,de_mirage
5,1-b3b89125-0263-417b-a51b-fe0b152cfcf9,1-b3b89125-0263-417b-a51b-fe0b152cfcf9,1776978796,1776980216,"{'winner': 'faction2', 'score': {'faction1': 4...",stage_1_20260424_0830,NaN,[{'player_id': '663a71f4-97dc-4a40-a7ea-467c1e...,2275.0,[{'player_id': '7fa5f3c5-96fb-4e30-87bf-d84544...,2281.0,de_mirage
6,1-1334d197-1679-4365-a349-8d05d83e5e6f,1-1334d197-1679-4365-a349-8d05d83e5e6f,1775611768,1775614182,"{'winner': 'faction1', 'score': {'faction1': 1...",stage_1_20260424_0830,NaN,[{'player_id': '663a71f4-97dc-4a40-a7ea-467c1e...,2341.0,[{'player_id': 'ac2a0721-1bef-44cf-9651-4dba79...,2344.0,de_ancient
7,1-f0ae6dca-f5cb-444f-8e54-5e771f81837d,1-f0ae6dca-f5cb-444f-8e54-5e771f81837d,1775499657,1775501725,"{'winner': 'faction2', 'score': {'faction1': 8...",stage_1_20260424_0830,NaN,[{'player_id': '663a71f4-97dc-4a40-a7ea-467c1e...,2458.0,[{'player_id': 'fc2eca9f-3394-40a5-89cc-d08c9e...,2480.0,de_anubis
8,1-f93b4761-129e-423e-8c03-0afc8dc4c3b7,1-f93b4761-129e-423e-8c03-0afc8dc4c3b7,1776046050,1776047826,"{'winner': 'faction1', 'score': {'faction1': 1...",stage_1_20260424_0830,NaN,[{'player_id': '4d6ac2f7-de8f-4ee9-badc-0c1ce0...,2395.0,[{'player_id': 'f9a55830-fd0f-4cb0-b8e5-167046...,2306.0,de_dust2
9,1-3294af3c-0c64-4692-b73e-513361c121d4,1-3294af3c-0c64-4692-b73e-513361c121d4,1776024939,1776026732,"{'winner': 'faction2', 'score': {'faction1': 6...",stage_1_20260424_0830,NaN,[{'player_id': '663a71f4-97dc-4a40-a7ea-467c1e...,2300.0,[{'player_id': '0bbfa577-501e-4095-97e3-2001c0...,2184.0,de_dust2


In [3]:
import pandas as pd

with pl.Config(tbl_rows=100, tbl_cols=200, tbl_width_chars=10000):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    display(pl.scan_parquet('data/interim/silver_ratings.parquet').head(100).collect().to_pandas())

,_id,batchId,Deaths_1,MVPs_1,Sniper Kill Rate per Round_1,ADR_1,Kills_1,Utility Successes_1,1v1Wins_1,Match Entry Success Rate_1,Match Entry Rate_1,Flash Successes_1,Sniper Kills_1,Enemies Flashed_1,K/D Ratio_1,Flashes per Round in a Match_1,First Kills_1,Entry Count_1,Pistol Kills_1,Result_1,Entry Wins_1,Utility Count_1,Utility Damage Success Rate per Match_1,Utility Damage_1,Utility Usage per Round_1,Zeus Kills_1,Headshots_1,Damage_1,1v2Count_1,Match 1v2 Win Rate_1,1v2Wins_1,Utility Enemies_1,Knife Kills_1,Enemies Flashed per Round in a Match_1,K/R Ratio_1,Match 1v1 Win Rate_1,Headshots %_1,Assists_1,Clutch Kills_1,1v1Count_1,Sniper Kill Rate per Match_1,Flash Count_1,Flash Success Rate per Match_1,Utility Success Rate per Match_1,Triple Kills_1,Quadro Kills_1,Utility Damage per Round in a Match_1,Penta Kills_1,Double Kills_1,Overtime score_1,Second Half Score_1,First Half Score_1,Team Win_1,Final Score_1,Team Headshots_1,Flash Count_2,Entry Count_2,Enemies Flashed per Round in a Match_2,Pistol Kills_2,Headshots %_2,Assists_2,Sniper Kills_2,Utility Enemies_2,Headshots_2,Flashes per Round in a Match_2,Quadro Kills_2,Flash Successes_2,ADR_2,Utility Successes_2,K/D Ratio_2,First Kills_2,Kills_2,Utility Damage per Round in a Match_2,Utility Count_2,Entry Wins_2,Utility Damage Success Rate per Match_2,K/R Ratio_2,Deaths_2,Sniper Kill Rate per Match_2,Result_2,Utility Success Rate per Match_2,Utility Damage_2,Match 1v2 Win Rate_2,Knife Kills_2,Double Kills_2,Match Entry Success Rate_2,Triple Kills_2,1v1Count_2,Damage_2,1v2Count_2,Utility Usage per Round_2,1v2Wins_2,Sniper Kill Rate per Round_2,Penta Kills_2,Enemies Flashed_2,Flash Success Rate per Match_2,Match Entry Rate_2,Zeus Kills_2,Clutch Kills_2,1v1Wins_2,MVPs_2,Match 1v1 Win Rate_2,Team Win_2,Team Headshots_2,First Half Score_2,Final Score_2,Overtime score_2,Second Half Score_2
0,1-4cb251f5-febf-41b1-808b-c94ae3882c74,NaN,19.799999,3.2,0.1320,79.000000,22.400000,5.80,0.0,0.608,0.1980,5.40,4.0,10.800000,1.1740,0.406,3.80,6.0,2.40,1.0,3.8,14.8,13.348000,204.399994,0.4940,0.0,10.4,2370.199951,0.40,0.000,0.0,7.00,0.0,0.3620,0.7460,0.000,48.400002,8.00,0.80,0.60,0.136,12.200000,0.4240,0.394,0.60,0.0,6.814000,0.0,5.40,4,5,7,1,16,10.4,15.600000,6.0,0.528,3.6,49.599998,6.2,2.2,2.6,9.8,0.520,0.4,8.6,72.599998,2.4,0.884,2.2,19.799999,1.952000,11.000000,2.2,5.962000,0.658,22.400000,0.120,0.0,0.232,58.599998,0.100,0.0,3.6,0.360,0.4,0.4,2178.000000,1.2,0.368,0.2,0.074,0.0,15.8,0.558,0.200,0.0,2.4,0.4,2.8,0.400,0,9.8,5,14,2,7
1,1-d8ccf3d9-326f-4656-853b-37fbd2dce3d5,NaN,15.800000,1.6,0.0200,71.139999,13.400000,1.20,0.0,0.304,0.2020,3.00,0.4,5.000000,0.8640,0.238,1.60,4.2,3.60,0.0,1.6,7.0,8.800000,54.200001,0.3340,0.0,7.2,1493.599976,0.60,0.200,0.2,1.20,0.0,0.2400,0.6380,0.000,55.799999,3.80,1.40,0.40,0.030,5.000000,0.5260,0.168,0.60,0.0,2.582000,0.2,2.40,0,2,6,0,8,7.2,5.000000,4.2,0.162,3.2,62.799999,3.4,0.6,3.2,10.2,0.242,0.2,2.0,82.440002,3.0,1.212,2.6,16.000000,3.600000,8.200000,2.6,8.682000,0.760,13.800000,0.032,1.0,0.412,75.599998,0.066,0.0,2.6,0.536,1.0,0.4,1731.400024,0.8,0.392,0.2,0.028,0.0,3.4,0.540,0.200,0.0,1.4,0.2,2.6,0.100,1,10.2,6,13,0,7
2,1-3421873c-e712-4be3-bdc0-818a4f183bc7,NaN,16.000000,2.0,0.0360,71.080002,14.400000,4.40,0.0,0.500,0.2000,6.00,0.8,7.200000,0.9080,0.524,2.80,4.6,3.00,0.0,2.8,11.8,8.032000,99.800003,0.5140,0.0,7.4,1635.000000,0.60,0.000,0.0,5.60,0.0,0.3120,0.6260,0.000,48.200001,3.40,0.60,0.00,0.064,12.000000,0.5000,0.346,0.40,0.2,4.340000,0.0,3.40,0,2,8,0,10,7.4,7.600000,4.6,0.192,3.0,48.799999,5.6,0.8,8.8,8.6,0.330,0.8,3.2,77.260002,5.4,1.118,1.8,16.000000,8.610001,12.000000,1.8,20.446001,0.696,14.400000,0.046,1.0,0.562,198.000000,0.000,0.0,2.8,0.258,0.4,0.0,1777.000000,0.6,0.520,0.0,0.034,0.0,4.4,0.392,0.198,0.0,0.6,0.0,2.6,0.000,1,8.6,4,13,0,9
3,1-1cb55b79-4ef3-4cfd-b89b-1cebeeab66ff,NaN,15.400000,1.0,0.0240,63.020000,9.400000,1.80,0.0,0.276,0.2020,4.60,0.4,6.800000,0.6040,0.588,1.20,3.6,3.00,0.0,1.2,3.2,9.060000,33.000000,0.1780,0.0,6.2,1134.599976,